# C00 — Stack, arquitetura e ferramentas do EduPredict Analytics

**Caderno reproduzível — inventário do repositório**, para a seção de Materiais e Métodos do artigo v0.3.

Documento de análise e referências: [`docs/cadernos/C00-stack-arquitetura-ferramentas.md`](../docs/cadernos/C00-stack-arquitetura-ferramentas.md). Não gera dados, não realiza treinos e não lê dados pessoais.

## 1. Inventário de dependências declaradas

A execução lê os manifestos locais de dependências, sem instalar pacotes ou chamar serviços externos. Faixas declaradas não equivalem às versões executadas.

In [ ]:
from pathlib import Path
import sys, json, platform, subprocess, importlib.metadata as metadata
import tomllib

root = Path.cwd().resolve()
if not (root / 'pyproject.toml').exists():
    raise FileNotFoundError('Execute este notebook na raiz do repositório EPA-cy')
python_config = tomllib.loads((root / 'pyproject.toml').read_text(encoding='utf-8'))
rust_config = tomllib.loads((root / 'engines/genesis-rs/Cargo.toml').read_text(encoding='utf-8'))
print('Python package:', python_config['project']['name'], python_config['project']['version'])
print('Python require:', python_config['project']['requires-python'])
print('Python optional groups:', python_config['project'].get('optional-dependencies', {}))
print('Rust crate:', rust_config['package']['name'], rust_config['package']['version'])
print('Rust deps:', rust_config['dependencies'])

## 2. Versões do ambiente local

Registra o que estiver efetivamente instalado. Ausências são registradas como `not-installed` e não constituem falha.

In [ ]:
packages = ['numpy', 'scikit-learn', 'duckdb', 'pyarrow', 'psutil', 'pytest', 'mlflow', 'xgboost']
installed = {}
for package in packages:
    try:
        installed[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed[package] = 'not-installed'
print(json.dumps({'python': sys.version.split()[0], 'os': platform.platform(), 'packages': installed}, indent=2))

## 3. Compiladores e runtime

Esta inspeção não executa `cargo build`, Rails, instalações ou migrações de banco.

In [ ]:
def tool_version(command):
    try:
        process = subprocess.run(command, capture_output=True, text=True, timeout=8, check=False)
        return (process.stdout or process.stderr).strip().splitlines()[0]
    except (OSError, subprocess.TimeoutExpired, IndexError):
        return 'not-available'
for cmd in [['rustc','--version'], ['cargo','--version'], ['ruby','--version'], ['bundle','--version']]:
    print(' '.join(cmd), '=>', tool_version(cmd))

## 4. Interfaces e contratos

- **ATLAS**: currículo sintético ilustrativo.
- **GENESIS**: geração Python e Rust, Parquet e manifestos SHA-256.
- **MAPA**: classificação triclasse N1/N2/N3 com divisão agrupada por estudante.
- **LUPA**: importância por permutação; não implica efeito causal.
- **Rails 8/Hotwire/SQLite**: laboratório local experimental; não é solução pública autenticada.

**Atenção ao contrato:** o leitor de snapshots MAPA `read_verified_v2` não lê diretamente os manifests Rust nativos. O adaptador de larga escala deve ser implementado e testado em cadernos posteriores.

## 5. Resultado para o artigo

O C00 sustenta a descrição de **Materiais e Métodos**. Os benchmarks Rust 25k, as auditorias SQL e a paridade ficam documentados como resultados de engenharia, não de qualidade preditiva.

Próximos cadernos: C01 GENESIS/escala, C02 datasets e C03 protocolo temporal. Manter referência ao commit, ao catálogo SHA-256 e às seeds em cada execução.